In [1]:
# Se precisar:  !pip install yfinance pandas
import yfinance as yf
import pandas as pd
from datetime import date

# === Carteira (na B3 o Yahoo Finance usa sufixo .SA) ===
tickers = ["PINE4","UGPA3","BHIA3","LIGT3","PETR4","CSAN3",
           "PRIO3","RAIL3","ODPV3","SAUD3","SCAR3","AZZA3"]

# (opcional) quantidades para rendimento PONDERADO da carteira.
# Deixe {} para média simples (peso igual entre os papéis).
quantidades = {}        # ex.: {"PETR4":100, "PRIO3":50, "UGPA3":200}

# === Período: do dia 1 do mês atual até hoje ===
hoje   = date.today()
inicio = hoje.replace(day=1)

# === Fechamentos ajustados (já consideram dividendos/JCP = "rendeu") ===
yahoo = [t + ".SA" for t in tickers]
px = yf.download(yahoo, start=inicio, end=hoje + pd.Timedelta(days=1),
                 auto_adjust=True, progress=False)["Close"]
if isinstance(px, pd.Series):           # caso venha 1 ticker só
    px = px.to_frame()

# === Variação de cada papel: 1º pregão do mês -> último pregão ===
reg = []
for t in tickers:
    y = t + ".SA"
    s = px[y].dropna() if y in px.columns else pd.Series(dtype=float)
    if s.empty:
        reg.append({"Ticker": t, "Preço 1º dia": None, "Preço atual": None, "Rendimento %": None})
    else:
        ini, fim = s.iloc[0], s.iloc[-1]
        reg.append({"Ticker": t, "Preço 1º dia": round(ini, 2),
                    "Preço atual": round(fim, 2),
                    "Rendimento %": round((fim/ini - 1)*100, 2)})

tab = pd.DataFrame(reg).sort_values("Rendimento %", ascending=False,
                                    na_position="last").reset_index(drop=True)

# === Rendimento da carteira ===
ok = tab.dropna(subset=["Rendimento %"]).set_index("Ticker")
if quantidades:
    val = {t: quantidades.get(t, 0) * ok.loc[t, "Preço 1º dia"] for t in ok.index}
    total = sum(val.values())
    rend = sum(val[t] * ok.loc[t, "Rendimento %"] for t in ok.index) / total if total else float("nan")
    metodo = "ponderado pelo valor investido"
else:
    rend = ok["Rendimento %"].mean()
    metodo = "média simples (peso igual)"

# === Saída ===
print(tab.to_string(index=False))
if len(px.index):
    print(f"\nPeríodo: {px.index.min().date()}  ->  {px.index.max().date()}")
print(f"Rendimento da carteira ({metodo}): {rend:.2f}%")
faltam = tab.loc[tab["Rendimento %"].isna(), "Ticker"].tolist()
if faltam:
    print("Sem dados no Yahoo Finance (confira o ticker):", ", ".join(faltam))

c:\Users\Kaue Mandarino\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Kaue Mandarino\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Ticker  Preço 1º dia  Preço atual  Rendimento %
 UGPA3         26.04        26.60          2.15
 RAIL3         13.17        13.45          2.13
 AZZA3         17.05        17.34          1.70
 ODPV3         14.25        14.41          1.12
 SAUD3         14.25        14.41          1.12
 BHIA3          1.07         1.08          0.93
 CSAN3          3.70         3.72          0.54
 PETR4         37.83        37.96          0.34
 PRIO3         52.40        52.57          0.32
 PINE4         11.97        11.90         -0.58
 SCAR3         12.35        12.14         -1.70
 LIGT3          3.38         3.27         -3.25

Período: 2026-07-01  ->  2026-07-02
Rendimento da carteira (média simples (peso igual)): 0.40%
